# FracID Case Study: Identification of the Fractional-Order Chen System

This notebook reproduces the benchmark case study for **FracID**:
- **System**: Fractional Chen chaotic attractor with commensurate order $\alpha = 0.90$.
- **Equations**:
  $$\begin{aligned}
  {}^C D^\alpha x &= a(y - x) \\
  {}^C D^\alpha y &= (c - a)x - xz + cy \\
  {}^C D^\alpha z &= xy - bz
  \end{aligned}$$
  with nominal parameters $a=35.0, b=3.0, c=28.0$.
- **Measurement conditions**: 5% Gaussian noise added, $T=1.5$, step $h=0.005$.
- **Objectives**:
  1. Forward simulation using the Adams–Bashforth–Moulton (Diethelm) predictor-corrector scheme.
  2. Synthesizing noisy observational time-series.
  3. Estimating $(\alpha, a, b, c)$ via bound-constrained optimization and checking recovery error.
  4. Comparing fractional model vs. integer-order model ($\Delta\text{AIC}, \Delta\text{BIC}$).
  5. Estimating parameter uncertainty via residual bootstrap.
  6. Plotting loss sensitivity curve across candidate values of $\alpha$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from fracid.models import FractionalChen
from fracid.data import generate
from fracid.estimation import FitProblem, fit_least_squares
from fracid.diagnostics import (
    compare_fractional_vs_integer,
    compute_alpha_sensitivity,
    compute_relative_errors,
    run_bootstrap,
    plot_trajectory_matplotlib,
    plot_phase_portrait_matplotlib,
    plot_residuals_matplotlib
)

print("FracID modules loaded successfully!")


## 1. Ground Truth Simulation & Data Generation with 5% Noise

In [ ]:
# 1. Define true model: alpha = 0.90, parameters a=35, b=3, c=28
true_chen = FractionalChen(alpha=0.90, params={"a": 35.0, "b": 3.0, "c": 28.0})
print("True model formulation:")
print(true_chen.latex())

# 2. Generate noisy data with 5% Gaussian noise (fine_factor=2 to avoid inverse crime)
dataset = generate(
    model=true_chen,
    T=1.5,
    h=0.005,
    noise_percent=5.0,
    seed=42,
    fine_factor=2
)
print(f"Generated {dataset.n} observations across states {dataset.names}.")


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for i, name in enumerate(dataset.names):
    axes[i].scatter(dataset.t, dataset.y[:, i], s=10, alpha=0.5, label=f"Observed {name} (5% noise)")
    axes[i].plot(dataset.t, dataset.clean[:, i], 'k-', lw=1.5, label=f"Clean {name}")
    axes[i].set_ylabel(name, fontweight="bold")
    axes[i].legend(loc="upper right")
    axes[i].grid(True, linestyle="--", alpha=0.5)
axes[-1].set_xlabel("Time t", fontweight="bold")
fig.suptitle("Fractional Chen System — Synthetic Measurements", fontweight="bold")
plt.tight_layout()
plt.show()


## 2. Parameter & Order Estimation

In [ ]:
# Set up inverse problem
problem = FitProblem(
    model=true_chen,
    data=dataset,
    free=["a", "b", "c"],
    alpha_bounds=(0.5, 1.0)
)

print(f"Free unknowns: {problem.names}")
print(f"Lower bounds: {problem.lower}")
print(f"Upper bounds: {problem.upper}")

# Run Least-Squares Identification using L-BFGS-B (or Nelder-Mead)
frac_result = fit_least_squares(problem, method="l-bfgs-b", seed=42)

print("\n=== Identification Result (Fractional Model) ===")
print(f"Estimated alpha: {frac_result.alpha:.4f} (True: 0.9000)")
for k, v in frac_result.params.items():
    if k in problem.free:
        print(f"Estimated {k}: {v:.4f} (True: {true_chen.params[k]})")
print(f"RMSE: {frac_result.rmse:.4f}")
print(f"Runtime: {frac_result.runtime:.2f} seconds")


In [ ]:
rel_errors = compute_relative_errors(frac_result)
print("=== Percentage Relative Errors ===")
for k, err in rel_errors.items():
    print(f"  {k}: {err:.2f} %")


## 3. Fractional vs. Integer-Order Model Comparison

In [ ]:
# Fit integer-order counterpart (alpha = 1.0 fixed)
problem_int = problem.integer_order()
int_result = fit_least_squares(problem_int, method="l-bfgs-b", seed=42)

# Side-by-side comparison table
comp_df = compare_fractional_vs_integer(frac_result, int_result)
print(comp_df.to_string())
print(f"\nDecision: {comp_df.attrs['verdict']}")


## 4. Fitted Trajectories, Phase Space & Residuals

In [ ]:
# Trajectory Fit
fig_fit = plot_trajectory_matplotlib(frac_result)
plt.show()

# Phase Portrait
fig_phase = plot_phase_portrait_matplotlib(frac_result)
plt.show()

# Residual diagnostics
fig_res = plot_residuals_matplotlib(frac_result)
plt.show()


## 5. Loss Profile vs. Fractional Order α

In [ ]:
sens_df = compute_alpha_sensitivity(frac_result, alpha_range=(0.75, 1.0), n_points=26, mode="slice")

plt.figure(figsize=(7, 4))
plt.plot(sens_df["alpha"], sens_df["sse"], 'bo-', lw=1.8, markersize=4)
plt.axvline(frac_result.alpha, color='red', linestyle='--', label=f"Estimated α = {frac_result.alpha:.3f}")
plt.axvline(0.90, color='green', linestyle=':', label="True α = 0.900")
plt.xlabel("Fractional Order α", fontweight="bold")
plt.ylabel("Objective Loss (SSE)", fontweight="bold")
plt.title("Sensitivity Profile: Loss vs. Order α", fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Parameter Uncertainty via Residual Bootstrap

In [ ]:
boot_res = run_bootstrap(frac_result, n_boot=20, ci_level=0.95, method="l-bfgs-b", seed=42)
print("=== 95% Confidence Intervals from Bootstrap ===")
print(boot_res.summary.to_string())
